In [1]:
# COMPLETE BIRD PEST DETECTION SYSTEM

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import librosa
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pickle
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

C:\Users\chami\anaconda3\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


In [2]:
# ============================================================================
# 1. DATA PREPROCESSING FUNCTIONS
# ============================================================================

def load_and_pad_audio(file_path, target_sr=16000, target_duration=5.0):
    """
    Load audio file and pad/trim to target duration
    """
    try:
        # Load audio
        y, sr = librosa.load(file_path, sr=target_sr)
        
        # Calculate target length
        target_length = int(target_sr * target_duration)
        
        # Pad or trim audio
        if len(y) > target_length:
            y = y[:target_length]
        elif len(y) < target_length:
            y = np.pad(y, (0, target_length - len(y)), mode='constant')
        
        return y, target_sr
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

def extract_mel_energy(y, sr, n_mels=32, n_fft=1024, hop_length=512):
    """
    Extract mel frequency energy features
    """
    # Extract mel spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=y, 
        sr=sr, 
        n_mels=n_mels, 
        n_fft=n_fft, 
        hop_length=hop_length,
        fmin=50,
        fmax=8000
    )
    
    # Convert to dB scale
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalize to [0, 1]
    mel_normalized = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())
    
    return mel_normalized

def process_audio_dataset(data_path, target_sr=16000, target_duration=5.0, 
                         n_mels=32, n_fft=1024, hop_length=512):
    """
    Process entire audio dataset
    """
    print(f"🎵 Processing audio dataset from: {data_path}")
    
    # Get class names from folder structure
    class_names = sorted([d for d in os.listdir(data_path) 
                         if os.path.isdir(os.path.join(data_path, d))])
    
    print(f"📂 Found {len(class_names)} classes: {class_names}")
    
    # Collect file paths and labels
    file_paths = []
    labels = []
    
    for class_idx, class_name in enumerate(class_names):
        class_path = os.path.join(data_path, class_name)
        class_files = [f for f in os.listdir(class_path) 
                      if f.lower().endswith(('.wav', '.mp3', '.flac'))]
        
        for file_name in class_files:
            file_paths.append(os.path.join(class_path, file_name))
            labels.append(class_idx)
        
        print(f"   {class_name}: {len(class_files)} files")
    
    print(f"📊 Total files: {len(file_paths)}")
    
    # Process audio files
    features = []
    processed_labels = []
    
    print("🔄 Extracting mel energy features...")
    for file_path, label in tqdm(zip(file_paths, labels), total=len(file_paths)):
        # Load and pad audio
        y, sr = load_and_pad_audio(file_path, target_sr, target_duration)
        
        if y is not None:
            # Extract mel energy features
            mel_features = extract_mel_energy(y, sr, n_mels, n_fft, hop_length)
            
            # Add channel dimension (H, W, C)
            mel_features = np.expand_dims(mel_features, axis=-1)
            
            features.append(mel_features)
            processed_labels.append(label)
    
    # Convert to numpy arrays
    features = np.array(features)
    processed_labels = np.array(processed_labels)
    
    print(f"✅ Processed {len(features)} samples")
    print(f"📏 Features shape: {features.shape}")
    print(f"🏷️ Labels shape: {processed_labels.shape}")
    
    return features, processed_labels, class_names

def prepare_data_for_training(features, labels, test_size=0.2, val_size=0.2, random_state=42):
    """
    Split data into train/validation/test sets
    """
    print("🔧 Preparing data for training...")
    
    # First split: separate test set
    X_temp, X_test, y_temp, y_test = train_test_split(
        features, labels, test_size=test_size, random_state=random_state, stratify=labels
    )
    
    # Second split: separate train and validation
    val_size_adjusted = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size_adjusted, random_state=random_state, stratify=y_temp
    )
    
    # Convert to categorical
    num_classes = len(np.unique(labels))
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
    y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes)
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)
    
    print(f"   Training: {X_train.shape[0]} samples")
    print(f"   Validation: {X_val.shape[0]} samples") 
    print(f"   Test: {X_test.shape[0]} samples")
    
    return X_train, X_val, X_test, y_train_cat, y_val_cat, y_test_cat, y_train, y_val, y_test


In [3]:
# ============================================================================
# 2. MODEL ARCHITECTURES
# ============================================================================

class EnhancedMicroDSCBlock(layers.Layer):
    """
    Enhanced Microcontroller-optimized Depthwise Separable Convolution Block
    
    Harmonized with ablation study architecture:
    - Depthwise conv: NO bias (use_bias=False)
    - Pointwise conv: NO bias (use_bias=False) — matches SeparableConv2D(use_bias=False)
    - Single BatchNormalization after pointwise (not after depthwise)
    - ReLU6 activation (quantization-friendly, bounded [0, 6])
    
    """
    def __init__(self, filters, kernel_size=3, strides=1, use_residual=False, **kwargs):
        super(EnhancedMicroDSCBlock, self).__init__(**kwargs)
        self.filters = filters
        self.kernel_size = kernel_size
        self.strides = strides
        self.use_residual = use_residual
        
        # Depthwise convolution (NO bias)
        self.depthwise_conv = layers.DepthwiseConv2D(
            kernel_size=kernel_size,
            strides=strides,
            padding='same',
            use_bias=False,  # NO bias for microcontroller optimization
            kernel_initializer='he_normal'
        )
        
        # ReLU6 after depthwise (no BN here — matches ablation SeparableConv2D)
        self.relu1 = layers.ReLU(max_value=6.0)  # ReLU6 for quantization
        
        # Pointwise convolution (NO bias — matches ablation SeparableConv2D(use_bias=False))
        self.pointwise_conv = layers.Conv2D(
            filters=filters,
            kernel_size=1,
            strides=1,
            padding='same',
            use_bias=False,  # NO bias
            kernel_initializer='he_normal'
        )
        
        # Single Batch normalization + ReLU6 after pointwise
        self.bn = layers.BatchNormalization()
        self.relu2 = layers.ReLU(max_value=6.0)
        
        # Optional residual connection
        if use_residual:
            self.add_layer = layers.Add()
    
    def call(self, inputs, training=None):
        x = inputs
        
        # Depthwise convolution + ReLU6
        x = self.depthwise_conv(x)
        x = self.relu1(x)
        
        # Pointwise convolution + BN + ReLU6
        x = self.pointwise_conv(x)
        x = self.bn(x, training=training)
        
        # Residual connection (if applicable and dimensions match)
        if (self.use_residual and 
            x.shape[-1] == inputs.shape[-1] and 
            self.strides == 1):
            x = self.add_layer([x, inputs])
        
        x = self.relu2(x)
        
        return x

def build_enhanced_microdsc_model(input_shape, num_classes):
    """
    Build Enhanced MicroDSC model
    Matches ablation study Model 5 (Enhanced MicroDSC + GAP):
    - Filter scaling: 16 -> 32 -> 48 -> 64
    - No bias terms in conv layers
    - ReLU6 activation
    - Global Average Pooling (replaces Flatten + Dense)
    - Dropout(0.2) before final softmax
    """
    inputs = layers.Input(shape=input_shape)
    
    # Initial convolution
    x = layers.Conv2D(16, 3, strides=2, padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU(max_value=6.0)(x)
    
    # Enhanced MicroDSC blocks (3 blocks: 32, 48, 64)
    x = EnhancedMicroDSCBlock(32, use_residual=False)(x)
    x = layers.MaxPooling2D(2)(x)
    
    x = EnhancedMicroDSCBlock(48, use_residual=False)(x)
    x = layers.MaxPooling2D(2)(x)
    
    x = EnhancedMicroDSCBlock(64, use_residual=False)(x)
    x = layers.MaxPooling2D(2)(x)
    
    # Classification (Global Average Pooling - key optimization)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name='Enhanced_MicroDSC')
    return model

def build_standard_dsc_model(input_shape, num_classes):
    """
    Build Standard DSC model (Baseline)
    Matches ablation study Model 1:
    - Filter scaling: 16 -> 32 -> 64 -> 128
    - Bias terms in all layers (use_bias=True)
    - ReLU activation
    - Flatten + Dense(128) classification head
    """
    inputs = layers.Input(shape=input_shape)
    
    # Initial Convolution (16 filters)
    x = layers.Conv2D(16, (3, 3), strides=2, padding='same', use_bias=True)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    
    # Block 1 (32 filters)
    x = layers.SeparableConv2D(32, (3, 3), padding='same', use_bias=True)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Block 2 (64 filters)
    x = layers.SeparableConv2D(64, (3, 3), padding='same', use_bias=True)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Block 3 (128 filters)
    x = layers.SeparableConv2D(128, (3, 3), padding='same', use_bias=True)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Classification (traditional: Flatten + Dense)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name='Standard_DSC')
    return model

def build_traditional_cnn_model(input_shape, num_classes):
    """
    Build Traditional CNN model
    """
    inputs = layers.Input(shape=input_shape)
    
    # Traditional CNN layers
    x = layers.Conv2D(32, 3, strides=2, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    
    x = layers.Conv2D(512, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    
    # Dense layers
    x = layers.Flatten()(x)
    x = layers.Dense(1024, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name='Traditional_CNN')
    return model

In [4]:
# ============================================================================
# 3. TRAINING AND EVALUATION FUNCTIONS
# ============================================================================

def evaluate_model_comprehensive(model, X_test, y_test_cat, y_test, class_names):
    """
    Comprehensive model evaluation
    """
    # Get predictions
    y_pred_probs = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    # Per-class metrics
    precision_per_class = precision_score(y_test, y_pred, average=None, zero_division=0)
    recall_per_class = recall_score(y_test, y_pred, average=None, zero_division=0)
    f1_per_class = f1_score(y_test, y_pred, average=None, zero_division=0)
    
    # Per-class accuracy
    accuracy_per_class = []
    for i in range(len(class_names)):
        class_mask = (y_test == i)
        if np.sum(class_mask) > 0:
            class_acc = accuracy_score(y_test[class_mask], y_pred[class_mask])
        else:
            class_acc = 0.0
        accuracy_per_class.append(class_acc)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    return {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'accuracy_per_class': accuracy_per_class,
        'confusion_matrix': cm,
        'y_pred': y_pred,
        'y_pred_probs': y_pred_probs
    }

def train_single_model(model_creator, model_name, X_train, y_train, X_val, y_val, 
                      X_test, y_test_cat, y_test, class_names, run_num, epochs=30):
    """
    Train a single model for one run
    """
    print(f"🏋️ Training {model_name} - Run {run_num}")
    
    # Create fresh model
    model = model_creator()
    
    # Compile model
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Callbacks
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True
    )
    
    # Train model
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=32,
        callbacks=[early_stopping],
        verbose=0
    )
    
    # Evaluate model
    results = evaluate_model_comprehensive(model, X_test, y_test_cat, y_test, class_names)
    
    # Add training info
    results['model_name'] = model_name
    results['run'] = run_num
    results['history'] = history.history
    results['parameters'] = model.count_params()
    results['final_val_accuracy'] = max(history.history['val_accuracy'])
    
    print(f"   ✅ Accuracy: {results['accuracy']:.4f}, Parameters: {results['parameters']:,}")
    
    return model, results

def train_all_models(X_train, y_train, X_val, y_val, X_test, y_test_cat, y_test, 
                    class_names, num_runs=10, epochs=30):
    """
    Train all models for specified number of runs
    """
    print("🚀 TRAINING ALL MODELS")
    print("=" * 50)
    
    input_shape = X_train.shape[1:]
    num_classes = len(class_names)
    
    # Model creators
    model_creators = {
        'Enhanced_MicroDSC': lambda: build_enhanced_microdsc_model(input_shape, num_classes),
        'Standard_DSC': lambda: build_standard_dsc_model(input_shape, num_classes),
        'Traditional_CNN': lambda: build_traditional_cnn_model(input_shape, num_classes)
    }
    
    # Storage for results
    all_results = {name: [] for name in model_creators.keys()}
    best_models = {name: {'model': None, 'accuracy': 0, 'run': 0} for name in model_creators.keys()}
    
    # Train each model multiple times
    for model_name, model_creator in model_creators.items():
        print(f"\n📊 Training {model_name} ({num_runs} runs)")
        print("-" * 40)
        
        for run in range(1, num_runs + 1):
            # Set random seed for reproducibility
            tf.random.set_seed(42 + run)
            np.random.seed(42 + run)
            
            # Train model
            model, results = train_single_model(
                model_creator, model_name, X_train, y_train, X_val, y_val,
                X_test, y_test_cat, y_test, class_names, run, epochs
            )
            
            # Store results
            all_results[model_name].append(results)
            
            # Update best model
            if results['accuracy'] > best_models[model_name]['accuracy']:
                best_models[model_name]['model'] = model
                best_models[model_name]['accuracy'] = results['accuracy']
                best_models[model_name]['run'] = run
            else:
                # Clean up model if not best
                del model
                tf.keras.backend.clear_session()
    
    return all_results, best_models

In [5]:
# ============================================================================
# 4. MODEL QUANTIZATION
# ============================================================================

def quantize_enhanced_microdsc(model, X_train_sample):
    """
    Quantize the Enhanced MicroDSC model for deployment
    """
    print("🔧 Quantizing Enhanced MicroDSC model...")
    
    # Representative dataset for quantization
    def representative_data_gen():
        for i in range(min(100, len(X_train_sample))):
            yield [X_train_sample[i:i+1].astype(np.float32)]
    
    # Convert to TensorFlow Lite with quantization
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    
    quantized_model = converter.convert()
    
    # Save quantized model
    with open('enhanced_microdsc_quantized.tflite', 'wb') as f:
        f.write(quantized_model)
    
    print("✅ Quantized model saved as 'enhanced_microdsc_quantized.tflite'")
    print(f"📦 Quantized model size: {len(quantized_model) / 1024:.2f} KB")
    
    return quantized_model



In [6]:
# ============================================================================
# 5. RESULTS SAVING AND ANALYSIS
# ============================================================================

def save_overall_results(all_results, class_names):
    """
    Save overall classification performance metrics
    """
    print("💾 Saving overall results...")
    
    # Create DataFrame for all results
    all_data = []
    for model_name, results_list in all_results.items():
        for result in results_list:
            row = {
                'model': model_name,
                'run': result['run'],
                'accuracy': result['accuracy'],
                'precision_macro': result['precision_macro'],
                'recall_macro': result['recall_macro'],
                'f1_macro': result['f1_macro'],
                'precision_weighted': result['precision_weighted'],
                'recall_weighted': result['recall_weighted'],
                'f1_weighted': result['f1_weighted'],
                'parameters': result['parameters'],
                'final_val_accuracy': result['final_val_accuracy']
            }
            all_data.append(row)
    
    df = pd.DataFrame(all_data)
    df.to_csv('overall_results_all_runs.csv', index=False)
    print("✅ Saved: overall_results_all_runs.csv")
    
    return df

def save_enhanced_microdsc_per_class_results(all_results, class_names):
    """
    Save per-class performance metrics for Enhanced MicroDSC
    """
    print("💾 Saving Enhanced MicroDSC per-class results...")
    
    enhanced_results = all_results['Enhanced_MicroDSC']
    
    per_class_data = []
    for result in enhanced_results:
        for i, class_name in enumerate(class_names):
            row = {
                'run': result['run'],
                'class': class_name,
                'class_index': i,
                'accuracy': result['accuracy_per_class'][i],
                'precision': result['precision_per_class'][i],
                'recall': result['recall_per_class'][i],
                'f1_score': result['f1_per_class'][i]
            }
            per_class_data.append(row)
    
    df_per_class = pd.DataFrame(per_class_data)
    df_per_class.to_csv('enhanced_microdsc_per_class_results.csv', index=False)
    print("✅ Saved: enhanced_microdsc_per_class_results.csv")
    
    return df_per_class

def save_confusion_matrices(best_models, X_test, y_test_cat, y_test, class_names):
    """
    Save confusion matrix analysis for best models
    """
    print("💾 Saving confusion matrix analysis...")
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for idx, (model_name, model_info) in enumerate(best_models.items()):
        model = model_info['model']
        
        # Get predictions
        y_pred_probs = model.predict(X_test, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        
        # Generate confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        
        # Plot confusion matrix
        ax = axes[idx]
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=class_names, yticklabels=class_names, ax=ax)
        ax.set_title(f'{model_name}\nAccuracy: {model_info["accuracy"]:.4f}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        
        # Save individual confusion matrix
        np.savetxt(f'confusion_matrix_{model_name}.csv', cm, delimiter=',', fmt='%d')
    
    plt.tight_layout()
    plt.savefig('confusion_matrices_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Saved: confusion_matrices_comparison.png")
    print("✅ Saved individual confusion matrices as CSV files")

def save_statistical_analysis(all_results):
    """
    Save statistical analysis (mean ± std) for all metrics
    """
    print("💾 Saving statistical analysis...")
    
    # Calculate statistics for each model
    stats_data = []
    
    for model_name, results_list in all_results.items():
        # Extract metrics
        metrics = {
            'accuracy': [r['accuracy'] for r in results_list],
            'precision_macro': [r['precision_macro'] for r in results_list],
            'recall_macro': [r['recall_macro'] for r in results_list],
            'f1_macro': [r['f1_macro'] for r in results_list],
            'precision_weighted': [r['precision_weighted'] for r in results_list],
            'recall_weighted': [r['recall_weighted'] for r in results_list],
            'f1_weighted': [r['f1_weighted'] for r in results_list],
            'parameters': [r['parameters'] for r in results_list],
            'final_val_accuracy': [r['final_val_accuracy'] for r in results_list]
        }
        
        # Calculate mean and std
        for metric_name, values in metrics.items():
            row = {
                'model': model_name,
                'metric': metric_name,
                'mean': np.mean(values),
                'std': np.std(values),
                'min': np.min(values),
                'max': np.max(values),
                'mean_plus_std': f"{np.mean(values):.4f} ± {np.std(values):.4f}"
            }
            stats_data.append(row)
    
    df_stats = pd.DataFrame(stats_data)
    df_stats.to_csv('statistical_analysis_summary.csv', index=False)
    
    # Create a pivot table for better visualization
    pivot_table = df_stats.pivot_table(
        index='metric', 
        columns='model', 
        values='mean_plus_std', 
        aggfunc='first'
    )
    
    pivot_table.to_csv('statistical_analysis_table.csv')
    
    print("✅ Saved: statistical_analysis_summary.csv")
    print("✅ Saved: statistical_analysis_table.csv")
    
    # Print summary
    print("\n STATISTICAL SUMMARY:")
    print(pivot_table)
    
    return df_stats, pivot_table

def save_best_models(best_models):
    """
    Save the best version of each model type
    """
    print("Saving best models...")
    
    for model_name, model_info in best_models.items():
        model = model_info['model']
        accuracy = model_info['accuracy']
        run = model_info['run']
        
        # Save model
        model_filename = f'best_{model_name.lower()}_acc_{accuracy:.4f}_run_{run}.h5'
        model.save(model_filename)
        
        print(f"✅ Saved: {model_filename}")
    
    # Save model info
    model_info_data = []
    for model_name, info in best_models.items():
        model_info_data.append({
            'model': model_name,
            'best_accuracy': info['accuracy'],
            'best_run': info['run'],
            'parameters': info['model'].count_params()
        })
    
    df_model_info = pd.DataFrame(model_info_data)
    df_model_info.to_csv('best_models_info.csv', index=False)
    print("✅ Saved: best_models_info.csv")

In [7]:
# ============================================================================
# 6. MAIN EXECUTION FUNCTION
# ============================================================================

def run_complete_bird_detection_analysis(data_path, num_runs=10, epochs=30):
    """
    Run complete bird pest detection analysis
    """
    print(" COMPLETE BIRD PEST DETECTION ANALYSIS")
    print("=" * 60)
    
    # Step 1: Data preprocessing
    print("\n1️⃣ DATA PREPROCESSING")
    features, labels, class_names = process_audio_dataset(data_path)
    X_train, X_val, X_test, y_train, y_val, y_test_cat, y_train_orig, y_val_orig, y_test = prepare_data_for_training(features, labels)
    
    # Step 2: Train all models
    print(f"\n2️⃣ MODEL TRAINING ({num_runs} runs, {epochs} epochs each)")
    all_results, best_models = train_all_models(
        X_train, y_train, X_val, y_val, X_test, y_test_cat, y_test, 
        class_names, num_runs, epochs
    )
    
    # Step 3: Save best models
    print("\n3️⃣ SAVING BEST MODELS")
    save_best_models(best_models)
    
    # Step 4: Quantize Enhanced MicroDSC
    print("\n4️⃣ MODEL QUANTIZATION")
    enhanced_model = best_models['Enhanced_MicroDSC']['model']
    quantized_model = quantize_enhanced_microdsc(enhanced_model, X_train[:100])
    
    # Step 5: Save all results
    print("\n5️⃣ SAVING RESULTS AND ANALYSIS")
    
    # a) Overall results
    df_overall = save_overall_results(all_results, class_names)
    
    # b) Per-class results for Enhanced MicroDSC
    df_per_class = save_enhanced_microdsc_per_class_results(all_results, class_names)
    
    # c) Confusion matrix analysis
    save_confusion_matrices(best_models, X_test, y_test_cat, y_test, class_names)
    
    # d) Statistical analysis
    df_stats, pivot_table = save_statistical_analysis(all_results)
    
    print("\n ANALYSIS COMPLETE!")
    print("📁 All results saved to current directory")
    
    return {
        'all_results': all_results,
        'best_models': best_models,
        'class_names': class_names,
        'quantized_model': quantized_model,
        'statistical_summary': pivot_table
    }

In [ ]:
# ============================================================================
# 7. USAGE
# ============================================================================
print("✅ Complete Bird Pest Detection System loaded!")
print("🚀 Usage:")
print("   results = run_complete_bird_detection_analysis('F:/BirdsPest_DataSet')")
print("   # Or with custom parameters:")
print("   results = run_complete_bird_detection_analysis('F:/BirdsPest_DataSet', num_runs=10, epochs=30)")

In [ ]:
# 8. Run complete analysis
results = run_complete_bird_detection_analysis('F:/BirdsPest_DataSet')